# Image Classification Pipeline
Train folder: `data/Train`

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision matplotlib scikit-learn pillow

## 2. Label Assignment
Labels are derived from filename prefixes (RA3xx = medical, AA = group A, qwerty = group Q, numeric = group N).

In [ ]:
import os

INPUT_DIR = "data/Train/"
files = os.listdir(INPUT_DIR)
print(f"Total files found: {len(files)}")
print(files[:10])

In [ ]:
import cv2, numpy, matplotlib, PIL
print("✅ All libraries working!")
print(f"OpenCV version: {cv2.__version__}")

In [ ]:
import os

INPUT_DIR = "data/Train/"
valid_ext = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')

all_files = sorted([f for f in os.listdir(INPUT_DIR) 
                     if f.lower().endswith(valid_ext)])

print(f"Total valid images: {len(all_files)}")
print(f"First 5: {all_files[:5]}")

In [ ]:
import hashlib

hashes = {}
duplicates = []

for fname in all_files:
    with open(INPUT_DIR + fname, 'rb') as f:
        h = hashlib.md5(f.read()).hexdigest()
    if h in hashes:
        duplicates.append((fname, hashes[h]))
    else:
        hashes[h] = fname

print(f"Unique images: {len(hashes)}")
print(f"Duplicate images: {len(duplicates)}")

In [ ]:
import cv2
import numpy as np
from PIL import Image

def create_valid_mask(img_array, thresh=8):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    return (gray > thresh).astype(np.uint8) * 255

def remove_noise(img_array):
    return cv2.fastNlMeansDenoisingColored(img_array, None, 5, 5, 7, 21)

def apply_clahe_masked(img_array, mask, clip_limit=2.0, tile_grid=(8,8)):
    lab = cv2.cvtColor(img_array, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_enhanced = clahe.apply(l)
    lab_enhanced = cv2.merge([l_enhanced, a, b])
    enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)
    mask_3ch = cv2.merge([mask, mask, mask]) / 255.0
    return (enhanced * mask_3ch).astype(np.uint8)

def preprocess_single_image(filepath, thresh=8, clip_limit=2.0):
    img = Image.open(filepath).convert("RGB")
    arr = np.array(img)
    mask = create_valid_mask(arr, thresh=thresh)
    denoised = remove_noise(arr)
    enhanced = apply_clahe_masked(denoised, mask, clip_limit=clip_limit)
    valid_pct = float((mask > 0).sum() / mask.size * 100)
    return {'original': arr, 'mask': mask, 'denoised': denoised,
            'enhanced': enhanced, 'valid_pct': valid_pct}

print("✅ Functions ready")

In [ ]:
import matplotlib.pyplot as plt

sample_files = all_files[:3]
fig, axes = plt.subplots(3, 3, figsize=(12,12))

for i, fname in enumerate(sample_files):
    result = preprocess_single_image(INPUT_DIR + fname)
    axes[i,0].imshow(result['original']); axes[i,0].set_title(f'{fname}\nOriginal'); axes[i,0].axis('off')
    axes[i,1].imshow(result['mask'], cmap='gray'); axes[i,1].set_title(f'Mask ({result["valid_pct"]:.1f}%)'); axes[i,1].axis('off')
    axes[i,2].imshow(result['enhanced']); axes[i,2].set_title('Enhanced'); axes[i,2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import json
import time

OUTPUT_DIR = "outputs/"
os.makedirs(OUTPUT_DIR + "enhanced/", exist_ok=True)
os.makedirs(OUTPUT_DIR + "masks/", exist_ok=True)

metadata = []
start_time = time.time()

for idx, fname in enumerate(all_files):
    base = os.path.splitext(fname)[0]
    try:
        result = preprocess_single_image(INPUT_DIR + fname)
        
        Image.fromarray(result['enhanced']).save(OUTPUT_DIR + f"enhanced/{base}_enhanced.png")
        Image.fromarray(result['mask']).save(OUTPUT_DIR + f"masks/{base}_mask.png")
        np.save(OUTPUT_DIR + f"masks/{base}_mask.npy", result['mask'])
        
        metadata.append({
            'filename': fname,
            'shape': list(result['original'].shape),
            'valid_pixel_pct': round(result['valid_pct'], 1),
            'status': 'success'
        })
        
        if (idx+1) % 20 == 0 or (idx+1) == len(all_files):
            print(f"[{idx+1}/{len(all_files)}] ✅ {fname} | valid: {result['valid_pct']:.1f}%")
        
    except Exception as e:
        metadata.append({'filename': fname, 'status': 'failed', 'error': str(e)})
        print(f"[{idx+1}/{len(all_files)}] ❌ {fname} | Error: {e}")

elapsed = time.time() - start_time

with open(OUTPUT_DIR + "preprocessing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

success = sum(1 for m in metadata if m['status'] == 'success')
print(f"\n{'='*50}")
print(f"✅ Done! {success}/{len(all_files)} images processed")
print(f"⏱  Time taken: {elapsed:.1f} seconds")
print(f"📁 Saved to: {OUTPUT_DIR}")
print(f"{'='*50}")

In [ ]:
import numpy as np

valid_pcts = [m['valid_pixel_pct'] for m in metadata if m['status']=='success']

print("=== Valid Pixel % Statistics Across All Images ===")
print(f"Mean:   {np.mean(valid_pcts):.1f}%")
print(f"Min:    {np.min(valid_pcts):.1f}%")
print(f"Max:    {np.max(valid_pcts):.1f}%")
print(f"Median: {np.median(valid_pcts):.1f}%")

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(valid_pcts, bins=20, color='steelblue', edgecolor='black')
plt.xlabel('Valid Pixel Percentage (%)')
plt.ylabel('Number of Images')
plt.title(f'Distribution of Valid Pixel % Across {len(valid_pcts)} Images')
plt.axvline(np.mean(valid_pcts), color='red', linestyle='--', label=f'Mean: {np.mean(valid_pcts):.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import random

random_samples = random.sample(all_files, 5)
fig, axes = plt.subplots(5, 3, figsize=(12, 20))

for i, fname in enumerate(random_samples):
    result = preprocess_single_image(INPUT_DIR + fname)
    axes[i,0].imshow(result['original']); axes[i,0].set_title(f'{fname}\nOriginal'); axes[i,0].axis('off')
    axes[i,1].imshow(result['mask'], cmap='gray'); axes[i,1].set_title(f'Mask ({result["valid_pct"]:.1f}%)'); axes[i,1].axis('off')
    axes[i,2].imshow(result['enhanced']); axes[i,2].set_title('Enhanced'); axes[i,2].axis('off')

plt.suptitle('Random Sample Verification - Full Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
PATCH_SIZE = 256
STRIDE     = 256
MIN_VALID  = 0.7

PATCH_DIR = OUTPUT_DIR + "patches/"
os.makedirs(PATCH_DIR, exist_ok=True)

enhanced_files = sorted(os.listdir(OUTPUT_DIR + "enhanced/"))
print(f"Found {len(enhanced_files)} enhanced images to patch")

patch_count = 0
skipped     = 0

for fname in enhanced_files:
    img = Image.open(OUTPUT_DIR + "enhanced/" + fname).convert("RGB")
    arr = np.array(img)
    height, width = arr.shape[:2]
    
    # Skip images smaller than patch size
    if height < PATCH_SIZE or width < PATCH_SIZE:
        continue
    
    base = os.path.splitext(fname)[0]
    
    for y in range(0, height - PATCH_SIZE + 1, STRIDE):
        for x in range(0, width - PATCH_SIZE + 1, STRIDE):
            patch = arr[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            
            # Check valid pixel ratio (non-black)
            gray_patch = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY)
            valid_ratio = np.sum(gray_patch > 8) / (PATCH_SIZE * PATCH_SIZE)
            
            if valid_ratio < MIN_VALID:
                skipped += 1
                continue
            
            patch_fname = f"{base}_patch_{y:04d}_{x:04d}.png"
            Image.fromarray(patch).save(PATCH_DIR + patch_fname)
            patch_count += 1

print(f"\n✅ Saved:   {patch_count} patches")
print(f"⏭ Skipped: {skipped} patches (too much black border)")

In [ ]:
patch_files = sorted(os.listdir(PATCH_DIR))
sample = patch_files[::max(1, len(patch_files)//9)][:9]

fig, axes = plt.subplots(3, 3, figsize=(12,12))
for ax, fname in zip(axes.flatten(), sample):
    patch = Image.open(PATCH_DIR + fname)
    ax.imshow(patch)
    ax.set_title(fname[:25], fontsize=7)
    ax.axis('off')

plt.suptitle(f'Sample Patches from {patch_count} Total', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(f"Total source images:  {len(enhanced_files)}")
print(f"Total patches created: {patch_count}")
print(f"Average patches per image: {patch_count/len(enhanced_files):.1f}")

In [ ]:
sizes = []
for fname in enhanced_files[:10]:
    img = Image.open(OUTPUT_DIR + "enhanced/" + fname)
    sizes.append(img.size)

print("Sample image sizes:")
for fname, size in zip(enhanced_files[:10], sizes):
    print(f"  {fname}: {size}")

In [ ]:
import os

total_size = 0
for f in os.listdir(PATCH_DIR):
    total_size += os.path.getsize(PATCH_DIR + f)

print(f"Total patches: {len(os.listdir(PATCH_DIR))}")
print(f"Total disk size: {total_size / (1024*1024):.1f} MB")

TRAIN/TEST SPLITS


In [ ]:
import shutil
from sklearn.model_selection import train_test_split

patch_files = sorted(os.listdir(PATCH_DIR))
print(f"Total patches to split: {len(patch_files)}")

# Split 70/15/15
train, temp = train_test_split(patch_files, test_size=0.30, random_state=42)
val, test   = train_test_split(temp, test_size=0.50, random_state=42)

print(f"Train: {len(train)} patches")
print(f"Val:   {len(val)}   patches")
print(f"Test:  {len(test)}  patches")

# Create split folders
DATASET_DIR = "outputs/dataset/"
for split, files in [("train", train), ("val", val), ("test", test)]:
    split_dir = f"{DATASET_DIR}{split}/"
    os.makedirs(split_dir, exist_ok=True)
    for f in files:
        shutil.copy(PATCH_DIR + f, split_dir + f)

print("\n✅ Dataset split complete!")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

PATCH_DIR = "outputs/dataset/train/"
GAN_OUTPUT_DIR = "outputs/gan_generated/"
os.makedirs(GAN_OUTPUT_DIR, exist_ok=True)

class PatchDataset(Dataset):
    def __init__(self, patch_dir, img_size=128):
        self.files = sorted([f for f in os.listdir(patch_dir) 
                              if f.lower().endswith(('.png','.jpg'))])
        self.patch_dir = patch_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.patch_dir + self.files[idx]).convert("RGB")
        img = img.resize((self.img_size, self.img_size))
        arr = np.array(img).astype(np.float32) / 127.5 - 1.0
        tensor = torch.tensor(arr).permute(2, 0, 1)
        return tensor

dataset = PatchDataset(PATCH_DIR)
print(f"✅ GAN training pool: {len(dataset)} real patches")

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256 * 8 * 8),
            nn.ReLU(),
            nn.Unflatten(1, (256, 8, 8)),
            
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128), nn.ReLU(),
            
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64), nn.ReLU(),
            
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.BatchNorm2d(32), nn.ReLU(),
            
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )
    def forward(self, z):
        return self.model(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            
            nn.Flatten(),
            nn.Linear(256*8*8, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

print("✅ Generator & Discriminator defined")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

if device.type == "cpu":
    print("⚠️ No GPU detected — training will be SLOW. Consider reducing epochs/batch size.")

G = Generator().to(device)
D = Discriminator().to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
criterion = nn.BCELoss()

loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)
print(f"✅ Ready — {len(loader)} batches per epoch")

GENAI AUGMNETATION

In [ ]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# REDUCED for CPU feasibility
NUM_EPOCHS = 10
SUBSET_SIZE = 500

subset_indices = np.random.choice(len(dataset), min(SUBSET_SIZE, len(dataset)), replace=False)
subset_dataset = Subset(dataset, subset_indices)
loader = DataLoader(subset_dataset, batch_size=32, shuffle=True)

print(f"Training on {len(subset_dataset)} patches, {NUM_EPOCHS} epochs")
print(f"Batches per epoch: {len(loader)}")

G_losses, D_losses = [], []
start = time.time()

for epoch in range(NUM_EPOCHS):
    for real_batch in loader:
        real_batch = real_batch.to(device)
        bs = real_batch.size(0)
        
        real_labels = torch.ones(bs, 1).to(device)
        fake_labels = torch.zeros(bs, 1).to(device)
        
        opt_D.zero_grad()
        real_pred = D(real_batch)
        d_real_loss = criterion(real_pred, real_labels)
        
        z = torch.randn(bs, 100).to(device)
        fake_batch = G(z)
        fake_pred = D(fake_batch.detach())
        d_fake_loss = criterion(fake_pred, fake_labels)
        
        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        opt_D.step()
        
        opt_G.zero_grad()
        z = torch.randn(bs, 100).to(device)
        fake_batch = G(z)
        fake_pred = D(fake_batch)
        g_loss = criterion(fake_pred, real_labels)
        g_loss.backward()
        opt_G.step()
    
    G_losses.append(g_loss.item())
    D_losses.append(d_loss.item())
    
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | G: {g_loss.item():.4f} | D: {d_loss.item():.4f} | {elapsed:.0f}s elapsed")

print(f"\n✅ Training complete in {time.time()-start:.0f}s")

import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.plot(G_losses, label='Generator')
plt.plot(D_losses, label='Discriminator')
plt.legend(); plt.title('GAN Training Loss'); plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(4, 100).to(device)
    samples = G(z).cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(12,3))
for ax, img in zip(axes, samples):
    img = ((img + 1) * 127.5).clip(0,255).astype(np.uint8)
    img = np.transpose(img, (1,2,0))
    ax.imshow(img)
    ax.axis('off')
plt.suptitle('Current Generator Output (Epoch 10)')
plt.show()

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256 * 8 * 8),
            nn.ReLU(),
            nn.Unflatten(1, (256, 8, 8)),
            
            # Replace ConvTranspose2d with Upsample + Conv2d
            nn.Upsample(scale_factor=2, mode='nearest'),  # 8->16
            nn.Conv2d(256, 128, 3, 1, 1),
            nn.BatchNorm2d(128), nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='nearest'),  # 16->32
            nn.Conv2d(128, 64, 3, 1, 1),
            nn.BatchNorm2d(64), nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='nearest'),  # 32->64
            nn.Conv2d(64, 32, 3, 1, 1),
            nn.BatchNorm2d(32), nn.ReLU(),
            
            nn.Upsample(scale_factor=2, mode='nearest'),  # 64->128
            nn.Conv2d(32, 3, 3, 1, 1),
            nn.Tanh()
        )
    def forward(self, z):
        return self.model(z)

print("✅ Fixed Generator (no checkerboard artifacts)")

In [ ]:
# Reinitialize with FIXED generator
G = Generator().to(device)
D = Discriminator().to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=0.0001, betas=(0.5, 0.999))

def smooth_real_labels(size):
    return torch.full((size, 1), 0.9).to(device)

criterion = nn.BCELoss()

NUM_EPOCHS = 20
G_losses, D_losses = [], []
start = time.time()

for epoch in range(NUM_EPOCHS):
    for real_batch in loader:
        real_batch = real_batch.to(device)
        bs = real_batch.size(0)
        
        real_labels = smooth_real_labels(bs)
        fake_labels = torch.zeros(bs, 1).to(device)
        
        opt_D.zero_grad()
        real_pred = D(real_batch)
        d_real_loss = criterion(real_pred, real_labels)
        
        z = torch.randn(bs, 100).to(device)
        fake_batch = G(z)
        fake_pred = D(fake_batch.detach())
        d_fake_loss = criterion(fake_pred, fake_labels)
        
        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        opt_D.step()
        
        for _ in range(2):
            opt_G.zero_grad()
            z = torch.randn(bs, 100).to(device)
            fake_batch = G(z)
            fake_pred = D(fake_batch)
            g_loss = criterion(fake_pred, torch.ones(bs,1).to(device))
            g_loss.backward()
            opt_G.step()
    
    G_losses.append(g_loss.item())
    D_losses.append(d_loss.item())
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | G: {g_loss.item():.4f} | D: {d_loss.item():.4f} | {elapsed:.0f}s")

print(f"\n✅ Done in {time.time()-start:.0f}s")

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(8, 100).to(device)
    samples = G(z).cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(14,7))
for ax, img in zip(axes.flatten(), samples):
    img = ((img + 1) * 127.5).clip(0,255).astype(np.uint8)
    img = np.transpose(img, (1,2,0))
    ax.imshow(img)
    ax.axis('off')
plt.suptitle('Generator Output After Fix (Epoch 20)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
G.eval()
NUM_TO_GENERATE = 200

with torch.no_grad():
    for i in range(NUM_TO_GENERATE):
        z = torch.randn(1, 100).to(device)
        fake = G(z).squeeze().cpu().numpy()
        fake = ((fake + 1) * 127.5).clip(0, 255).astype(np.uint8)
        fake = np.transpose(fake, (1, 2, 0))
        Image.fromarray(fake).save(f"{GAN_OUTPUT_DIR}gan_patch_{i:04d}.png")

print(f"✅ Generated {NUM_TO_GENERATE} synthetic patches → {GAN_OUTPUT_DIR}")

In [ ]:
torch.save(G.state_dict(), "outputs/gan_generator_final.pth")
torch.save(D.state_dict(), "outputs/gan_discriminator_final.pth")
print("✅ Models saved — can reload later without retraining")

In [ ]:
import shutil

# Version A - Baseline (no augmentation) - already have this
# Version B - Roboflow standard augmentation - export from Roboflow
# Version C - GAN-augmented

VERSION_C_DIR = "outputs/dataset_v3_gan_augmented/"
os.makedirs(VERSION_C_DIR, exist_ok=True)

# Copy real training patches
for f in os.listdir("outputs/dataset/train/"):
    shutil.copy(f"outputs/dataset/train/{f}", VERSION_C_DIR + f)

# Add GAN-generated patches
for f in os.listdir(GAN_OUTPUT_DIR):
    shutil.copy(GAN_OUTPUT_DIR + f, VERSION_C_DIR + f)

total = len(os.listdir(VERSION_C_DIR))
print(f"✅ Version C (GAN-augmented) dataset: {total} patches")
print(f"   Real: {len(os.listdir('outputs/dataset/train/'))}")
print(f"   GAN synthetic: {len(os.listdir(GAN_OUTPUT_DIR))}")

In [ ]:
G.eval()
NUM_TO_GENERATE = 1000

print(f"Generating {NUM_TO_GENERATE} synthetic patches...")
start = time.time()

with torch.no_grad():
    for i in range(NUM_TO_GENERATE):
        z = torch.randn(1, 100).to(device)
        fake = G(z).squeeze().cpu().numpy()
        fake = ((fake + 1) * 127.5).clip(0, 255).astype(np.uint8)
        fake = np.transpose(fake, (1, 2, 0))
        Image.fromarray(fake).save(f"{GAN_OUTPUT_DIR}gan_patch_{i:04d}.png")
        
        if (i+1) % 200 == 0:
            print(f"  {i+1}/{NUM_TO_GENERATE} generated...")

elapsed = time.time() - start
print(f"\n✅ Generated {NUM_TO_GENERATE} synthetic patches in {elapsed:.0f}s")
print(f"📁 Saved to: {GAN_OUTPUT_DIR}")

In [ ]:
import shutil

VERSION_C_DIR = "outputs/dataset_v3_gan_augmented/"

# Clear and rebuild fresh
if os.path.exists(VERSION_C_DIR):
    shutil.rmtree(VERSION_C_DIR)
os.makedirs(VERSION_C_DIR, exist_ok=True)

# Copy real training patches
for f in os.listdir("outputs/dataset/train/"):
    shutil.copy(f"outputs/dataset/train/{f}", VERSION_C_DIR + f)

# Add all GAN-generated patches
for f in os.listdir(GAN_OUTPUT_DIR):
    shutil.copy(GAN_OUTPUT_DIR + f, VERSION_C_DIR + f)

total = len(os.listdir(VERSION_C_DIR))
real_count = len(os.listdir('outputs/dataset/train/'))
gan_count = len(os.listdir(GAN_OUTPUT_DIR))

print(f"✅ Version C (GAN-augmented) rebuilt:")
print(f"   Real:         {real_count} patches")
print(f"   GAN synthetic: {gan_count} patches ({gan_count/total*100:.1f}% of total)")
print(f"   Total:        {total} patches")

THIS MORE PATCHES GENERATED DATSET BY GENRATOR


In [ ]:
torch.save(G.state_dict(), "outputs/gan_generator_final.pth")
torch.save(D.state_dict(), "outputs/gan_discriminator_final.pth")
print("✅ Models saved permanently to outputs/")

In [ ]:
import random
import shutil
import os

ANNOTATION_DIR = "outputs/to_annotate/"
os.makedirs(ANNOTATION_DIR, exist_ok=True)

train_patches = os.listdir("outputs/dataset/train/")

SAMPLE_SIZE = 150  # realistic for manual annotation workload
random.seed(42)
sample_for_annotation = random.sample(train_patches, min(SAMPLE_SIZE, len(train_patches)))

for f in sample_for_annotation:
    shutil.copy(f"outputs/dataset/train/{f}", ANNOTATION_DIR + f)

print(f"✅ {len(sample_for_annotation)} patches ready for Roboflow upload")
print(f"📁 Location: {ANNOTATION_DIR}")

MAAUGMNETATION 


In [ ]:
import os
import shutil
import random

train_patches = os.listdir("outputs/dataset/train/")
random.seed(42)

# Select 30 patches for manageable QGIS workload
SAMPLE_SIZE = 30
manual_sample = random.sample(train_patches, SAMPLE_SIZE)

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
os.makedirs(QGIS_LABEL_DIR, exist_ok=True)

for f in manual_sample:
    shutil.copy(f"outputs/dataset/train/{f}", QGIS_LABEL_DIR + f)

print(f"✅ {len(manual_sample)} patches copied to: {QGIS_LABEL_DIR}")
print("\nFiles to annotate:")
for f in manual_sample:
    print(f"  {f}")

QGIS WORK BEGINS

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(5, 6, figsize=(18,15))
for ax, fname in zip(axes.flatten(), manual_sample):
    img = Image.open(QGIS_LABEL_DIR + fname)
    ax.imshow(img)
    ax.set_title(fname[:20], fontsize=7)
    ax.axis('off')
plt.suptitle('30 Patches Selected for Manual QGIS Annotation', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/qgis_to_label/preview_grid.png', dpi=100)
plt.show()

Since images are cloudy,so make them enhance for final annotation

In [ ]:
import numpy as np
from PIL import Image
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
patches = os.listdir(QGIS_LABEL_DIR)
patches = [p for p in patches if p.endswith('.png')]

print("Checking for cloud-heavy patches (high brightness, low color variance)...")

cloud_suspects = []
for fname in patches:
    img = np.array(Image.open(QGIS_LABEL_DIR + fname).convert("RGB"))
    
    brightness = img.mean()
    saturation_proxy = img.std()  # clouds are flat/low-variance, bright
    
    if brightness > 180 and saturation_proxy < 35:
        cloud_suspects.append((fname, brightness, saturation_proxy))

print(f"\n⚠️ {len(cloud_suspects)} likely cloud-affected patches found:")
for fname, b, s in cloud_suspects:
    print(f"  {fname}: brightness={b:.1f}, variance={s:.1f}")

In [ ]:
import random

# Remove cloud-suspects from the working set
clean_patches = [p for p in patches if p not in [c[0] for c in cloud_suspects]]

# Get replacement patches (not already in our 30)
all_train_patches = os.listdir("outputs/dataset/train/")
remaining_pool = [p for p in all_train_patches if p not in patches]

random.seed(43)  # different seed for fresh picks
n_replacements = len(cloud_suspects)
replacements = random.sample(remaining_pool, n_replacements)

# Check replacements aren't ALSO cloudy (quick filter)
import shutil
final_replacements = []
for fname in replacements:
    img = np.array(Image.open(f"outputs/dataset/train/{fname}").convert("RGB"))
    if img.mean() <= 180 or img.std() >= 35:  # passes cloud filter
        final_replacements.append(fname)
        shutil.copy(f"outputs/dataset/train/{fname}", QGIS_LABEL_DIR + fname)

# Remove the cloudy ones from the folder
for fname, _, _ in cloud_suspects:
    os.remove(QGIS_LABEL_DIR + fname)

print(f"✅ Removed {len(cloud_suspects)} cloudy patches")
print(f"✅ Added {len(final_replacements)} clearer replacements")
print(f"📁 Final clean set: {len(os.listdir(QGIS_LABEL_DIR))} patches ready for annotation")

FRESH PREVIEW

In [ ]:
import matplotlib.pyplot as plt

final_patches = [f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')]

fig, axes = plt.subplots(5, 6, figsize=(18,15))
for ax, fname in zip(axes.flatten(), final_patches):
    img = Image.open(QGIS_LABEL_DIR + fname)
    ax.imshow(img)
    ax.set_title(fname[:20], fontsize=7)
    ax.axis('off')
plt.suptitle('Final 30 Clean Patches for QGIS Annotation', fontweight='bold')
plt.tight_layout()
plt.savefig(QGIS_LABEL_DIR + 'preview_grid_final.png', dpi=100)
plt.show()

CHECKINH OF ROW GRID IMAGE AS CORRUPTED

In [ ]:
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
files = os.listdir(QGIS_LABEL_DIR)

print("All files in annotation folder:")
for f in files:
    print(f"  {f}")

In [ ]:
import numpy as np
from PIL import Image
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"

# Exclude any preview/grid files explicitly
all_files = [f for f in os.listdir(QGIS_LABEL_DIR) 
             if f.endswith('.png') and 'preview' not in f.lower() and 'grid' not in f.lower()]

print(f"Checking {len(all_files)} actual patches...")

cloud_suspects = []
for fname in all_files:
    img = np.array(Image.open(QGIS_LABEL_DIR + fname).convert("RGB"))
    
    brightness = img.mean()
    variance = img.std()
    
    # Tighter, more aggressive cloud detection
    is_cloudy = (brightness > 160 and variance < 45) or (brightness > 200)
    
    if is_cloudy:
        cloud_suspects.append((fname, brightness, variance))

print(f"\n⚠️ {len(cloud_suspects)} cloud/haze-affected patches found:")
for fname, b, v in cloud_suspects:
    print(f"  {fname}: brightness={b:.1f}, variance={v:.1f}")

In [ ]:
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
all_files = sorted([f for f in os.listdir(QGIS_LABEL_DIR) 
                     if f.endswith('.png') and 'preview' not in f.lower()])

print(f"Total files: {len(all_files)}\n")
for i, f in enumerate(all_files, 1):
    print(f"{i:2d}. {f}")

Chechinf of some images


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fname = "RA307SEP2023035024011100054PSANSTUC00GTDF_Clahe_enhanced_patch_0256_2048.png"
img = Image.open(f"outputs/qgis_to_label/{fname}")

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.title(fname)
plt.axis('off')
plt.show()

print(f"Image size: {img.size}")
print(f"Image mode: {img.mode}")

In [ ]:
fname2 = "RA310JUL2024039386011000054PSANSTUC00GTDF_Clahe_enhanced_patch_1536_1792.png"
img2 = Image.open(f"outputs/qgis_to_label/{fname2}")

plt.figure(figsize=(8,8))
plt.imshow(img2)
plt.title(fname2)
plt.axis('off')
plt.show()

In [ ]:
suspicious_files = [
    "RA311FEB2025042455011000053PSANSTUC00GTDF_Clahe_enhanced_patch_1536_1280.png",
    "RA312DEC2023036388011100054PSANSTUC00GTDF_Clahe_enhanced_patch_1024_0256.png",
    "RA312MAR2025042867011100054PSANSTUC00GTDF 2_Clahe_enhanced_patch_0768_0512.png",
    "RA312NOV2024041162011100054PSANSTUC00GTDF_Clahe_enhanced_patch_0512_2048.png",
    "qwerty (28)_enhanced_patch_1280_0256.png"
]

fig, axes = plt.subplots(1, len(suspicious_files), figsize=(22,4))
for ax, fname in zip(axes, suspicious_files):
    img = Image.open(f"outputs/qgis_to_label/{fname}")
    ax.imshow(img)
    ax.set_title(fname[:30], fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import random
import shutil
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"

# Confirmed discards
to_remove = [
    "RA310JUL2024039386011000054PSANSTUC00GTDF_Clahe_enhanced_patch_1536_1792.png",
    "qwerty (28)_enhanced_patch_1280_0256.png"
]

# Remove them
for fname in to_remove:
    path = QGIS_LABEL_DIR + fname
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed: {fname}")

# Get current set + pool for replacements
current_files = set(os.listdir(QGIS_LABEL_DIR))
all_train_patches = os.listdir("outputs/dataset/train/")
remaining_pool = [p for p in all_train_patches if p not in current_files]

random.seed(99)
replacements = random.sample(remaining_pool, len(to_remove))

for fname in replacements:
    shutil.copy(f"outputs/dataset/train/{fname}", QGIS_LABEL_DIR + fname)
    print(f"Added: {fname}")

print(f"\n✅ Final set: {len([f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')])} patches")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(replacements), figsize=(10,5))
if len(replacements) == 1:
    axes = [axes]
for ax, fname in zip(axes, replacements):
    img = Image.open(QGIS_LABEL_DIR + fname)
    ax.imshow(img)
    ax.set_title(fname[:25], fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
all_files = sorted([f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')])

print(f"Total files: {len(all_files)}\n")
for i, f in enumerate(all_files, 1):
    print(f"{i:2d}. {f}")

In [ ]:
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"

# Remove the 2 non-patch preview files
non_patches = ["preview_grid.png", "preview_grid_final.png"]

for fname in non_patches:
    path = QGIS_LABEL_DIR + fname
    if os.path.exists(path):
        os.remove(path)
        print(f"✅ Removed: {fname}")

# Also remove the confirmed cloud patch we identified
cloud_patch = "qwerty (61)_enhanced_patch_1024_0512.png"
path = QGIS_LABEL_DIR + cloud_patch
if os.path.exists(path):
    os.remove(path)
    print(f"✅ Removed: {cloud_patch}")

# Verify final count
remaining = [f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')]
print(f"\nCurrent count: {len(remaining)}")

In [ ]:
import random
import shutil

current_files = set(os.listdir(QGIS_LABEL_DIR))
all_train_patches = os.listdir("outputs/dataset/train/")
remaining_pool = [p for p in all_train_patches if p not in current_files]

random.seed(200)
new_replacement = random.sample(remaining_pool, 1)[0]
shutil.copy(f"outputs/dataset/train/{new_replacement}", QGIS_LABEL_DIR + new_replacement)
print(f"✅ Added: {new_replacement}")

final_count = len([f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')])
print(f"\n✅ Final clean count: {final_count}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(QGIS_LABEL_DIR + new_replacement)
plt.figure(figsize=(6,6))
plt.imshow(img)
plt.title(new_replacement)
plt.axis('off')
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

final_patches = sorted([f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')])
print(f"Total patches ready: {len(final_patches)}")

fig, axes = plt.subplots(5, 6, figsize=(18,15))
for ax, fname in zip(axes.flatten(), final_patches):
    img = Image.open(QGIS_LABEL_DIR + fname)
    ax.imshow(img)
    ax.set_title(fname[:22], fontsize=6)
    ax.axis('off')

plt.suptitle('Final 30 Verified Patches — Ready for QGIS Annotation', fontweight='bold')
plt.tight_layout()
plt.savefig(QGIS_LABEL_DIR + 'final_verified_grid.png', dpi=100)
plt.show()

In [ ]:
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
final_files = sorted([f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')])

print(f"Total files: {len(final_files)}\n")
for i, f in enumerate(final_files, 1):
    print(f"{i:2d}. {f}")

In [ ]:
import os

QGIS_LABEL_DIR = "outputs/qgis_to_label/"

# Remove the non-patch grid visualization file
grid_file = "final_verified_grid.png"
path = QGIS_LABEL_DIR + grid_file

if os.path.exists(path):
    os.remove(path)
    print(f"✅ Removed: {grid_file}")
else:
    print("File not found (already removed?)")

# Verify final count
remaining = [f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')]
print(f"\n✅ Final count: {len(remaining)} patches")

In [ ]:
import os
import json

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
final_files = sorted([f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')])

patch_id_map = {i+1: fname for i, fname in enumerate(final_files)}

# Save as JSON for later use (COCO conversion will need this)
with open("outputs/qgis_annotations/patch_id_mapping.json", "w") as f:
    json.dump(patch_id_map, f, indent=2)

print("✅ Mapping saved to outputs/qgis_annotations/patch_id_mapping.json")
for pid, fname in patch_id_map.items():
    print(f"{pid:2d} → {fname}")

In [ ]:
import os
import json

# Create the folder first
os.makedirs("outputs/qgis_annotations/", exist_ok=True)

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
final_files = sorted([f for f in os.listdir(QGIS_LABEL_DIR) if f.endswith('.png')])

patch_id_map = {i+1: fname for i, fname in enumerate(final_files)}

with open("outputs/qgis_annotations/patch_id_mapping.json", "w") as f:
    json.dump(patch_id_map, f, indent=2)

print("✅ Mapping saved to outputs/qgis_annotations/patch_id_mapping.json")
for pid, fname in patch_id_map.items():
    print(f"{pid:2d} → {fname}")

In [ ]:
import os
import json

VIA_JSON_PATH = "outputs/qgis_annotations/via_export_FINAL_MERGED.json"
IMAGE_DIR = "QGIS_Annotation_Patches/"

print("File exists:", os.path.exists(VIA_JSON_PATH))
print("File size:", os.path.getsize(VIA_JSON_PATH), "bytes")

with open(VIA_JSON_PATH) as f:
    via_data = json.load(f)

img_metadata = via_data["_via_img_metadata"]
print(f"Images in JSON: {len(img_metadata)}")

missing = []
for img_key, img_info in img_metadata.items():
    fname = img_info["filename"]
    path = IMAGE_DIR + fname
    if not os.path.exists(path):
        missing.append(fname)

print(f"\nImages found on disk: {len(img_metadata) - len(missing)}")
print(f"Images MISSING: {len(missing)}")

if missing:
    print("\n⚠️ Missing files:")
    for m in missing:
        print(f"   {m}")

In [ ]:
import os

# Check if QGIS_Annotation_Patches exists relative to current directory
print("Current working directory:", os.getcwd())
print("QGIS_Annotation_Patches exists:", os.path.exists("QGIS_Annotation_Patches/"))

if os.path.exists("QGIS_Annotation_Patches/"):
    files = os.listdir("QGIS_Annotation_Patches/")
    print(f"Files inside: {len(files)}")
    print(files[:5])

In [ ]:
import os

# List everything in your project root to see actual folder names
print("Contents of project root:")
for item in os.listdir("."):
    print(f"  {item}")

In [ ]:
import os

# Check the actual patches folder
IMAGE_DIR = "outputs/qgis_to_label/"

print("Folder exists:", os.path.exists(IMAGE_DIR))

if os.path.exists(IMAGE_DIR):
    files = os.listdir(IMAGE_DIR)
    print(f"Total files: {len(files)}")
    print(f"First 5: {files[:5]}")

In [ ]:
import os
import json

VIA_JSON_PATH = "outputs/qgis_annotations/via_export_FINAL_MERGED.json"
IMAGE_DIR = "outputs/qgis_to_label/"   # ← corrected path

print("Folder exists:", os.path.exists(IMAGE_DIR))
print("Files in folder:", len(os.listdir(IMAGE_DIR)))

with open(VIA_JSON_PATH) as f:
    via_data = json.load(f)

img_metadata = via_data["_via_img_metadata"]
print(f"Images in JSON: {len(img_metadata)}")

missing = []
for img_key, img_info in img_metadata.items():
    fname = img_info["filename"]
    path = IMAGE_DIR + fname
    if not os.path.exists(path):
        missing.append(fname)

print(f"\nImages found on disk: {len(img_metadata) - len(missing)}")
print(f"Images MISSING: {len(missing)}")

if missing:
    print("\n⚠️ Still missing:")
    for m in missing:
        print(f"   {m}")

COCO SCRIPTS

In [ ]:
import json
import os
from PIL import Image
import shutil
import math

# ── CONFIG (CORRECTED PATHS) ──────────────────────────────
VIA_JSON_PATH   = "outputs/qgis_annotations/via_export_FINAL_MERGED.json"
IMAGE_DIR       = "outputs/qgis_to_label/"
OUTPUT_COCO_DIR = "outputs/coco_dataset/"
# ────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_COCO_DIR, exist_ok=True)
os.makedirs(OUTPUT_COCO_DIR + "images/", exist_ok=True)

CLASS_NAMES = {
    1: "Cropland", 2: "Fallow", 3: "Plantation", 4: "Sandy_area",
    5: "Scrub_land", 6: "Mining", 7: "Rural", 8: "Urban",
    9: "Deciduous", 10: "Evergreen", 11: "Forest_plantation",
    12: "Grass_grazing", 13: "Inland_wetland", 14: "Riverstream_canals",
    15: "Water_bodies"
}
VALID_CLASS_IDS = set(CLASS_NAMES.keys())

with open(VIA_JSON_PATH) as f:
    via_data = json.load(f)

img_metadata = via_data["_via_img_metadata"]
print(f"Loaded {len(img_metadata)} annotated images from VIA export")

coco_output = {
    "info": {"description": "Shillong LULC Detection Dataset", "version": "1.0"},
    "images": [],
    "annotations": [],
    "categories": [{"id": cid, "name": name, "supercategory": "land_cover"}
                    for cid, name in CLASS_NAMES.items()]
}

image_id_counter = 1
annotation_id_counter = 1
skipped_regions = 0
skipped_images = 0

for img_key, img_info in img_metadata.items():
    fname = img_info["filename"]
    img_path = IMAGE_DIR + fname

    if not os.path.exists(img_path):
        print(f"⚠️ WARNING: Image not found, skipping: {fname}")
        skipped_images += 1
        continue

    with Image.open(img_path) as im:
        width, height = im.size

    coco_output["images"].append({
        "id": image_id_counter, "file_name": fname,
        "width": width, "height": height
    })

    for region in img_info.get("regions", []):
        if region is None:
            skipped_regions += 1
            continue

        shape = region.get("shape_attributes", {})
        attrs = region.get("region_attributes", {})
        class_name_raw = attrs.get("class_name", "")

        try:
            class_id = int(class_name_raw)
        except (ValueError, TypeError):
            skipped_regions += 1
            continue

        if class_id not in VALID_CLASS_IDS:
            skipped_regions += 1
            continue

        shape_name = shape.get("name", "")

        if shape_name == "polygon":
            xs = shape.get("all_points_x", [])
            ys = shape.get("all_points_y", [])
            if not xs or not ys or any(v is None for v in xs + ys):
                skipped_regions += 1
                continue
            xs = [max(0, x) for x in xs]
            ys = [max(0, y) for y in ys]
            segmentation = [[c for pair in zip(xs, ys) for c in pair]]
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

        elif shape_name == "ellipse":
            cx, cy = shape.get("cx", 0), shape.get("cy", 0)
            rx, ry = shape.get("rx", 0), shape.get("ry", 0)
            n_points = 16
            xs, ys = [], []
            for i in range(n_points):
                angle = 2 * math.pi * i / n_points
                xs.append(cx + rx * math.cos(angle))
                ys.append(cy + ry * math.sin(angle))
            segmentation = [[c for pair in zip(xs, ys) for c in pair]]
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)
        else:
            skipped_regions += 1
            continue

        bbox = [x_min, y_min, x_max - x_min, y_max - y_min]
        area = (x_max - x_min) * (y_max - y_min)

        coco_output["annotations"].append({
            "id": annotation_id_counter, "image_id": image_id_counter,
            "category_id": class_id, "bbox": bbox, "area": area,
            "segmentation": segmentation, "iscrowd": 0
        })
        annotation_id_counter += 1

    image_id_counter += 1

print(f"\n{'='*60}")
print(f"✅ CONVERSION COMPLETE")
print(f"{'='*60}")
print(f"Images converted:   {len(coco_output['images'])}")
print(f"Valid annotations:  {len(coco_output['annotations'])}")
print(f"Skipped regions:    {skipped_regions}")
print(f"Skipped images:     {skipped_images}")

output_path = OUTPUT_COCO_DIR + "annotations_coco.json"
with open(output_path, "w") as f:
    json.dump(coco_output, f, indent=2)
print(f"\n✅ Saved: {output_path}")

copied = 0
for img_entry in coco_output["images"]:
    src = IMAGE_DIR + img_entry["file_name"]
    dst = OUTPUT_COCO_DIR + "images/" + img_entry["file_name"]
    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1
print(f"✅ Copied {copied} images to: {OUTPUT_COCO_DIR}images/")

class_dist = {}
for ann in coco_output["annotations"]:
    cid = ann["category_id"]
    class_dist[cid] = class_dist.get(cid, 0) + 1

print(f"\n=== Final Class Distribution ===")
for cid in sorted(class_dist.keys()):
    print(f"  {CLASS_NAMES[cid]:25s} (id {cid:2d}): {class_dist[cid]:3d} annotations")

print(f"\n✅ READY FOR MASK R-CNN TRAINING")

In [ ]:
# Cell 1
import subprocess
subprocess.run(["pip", "install", "torch", "torchvision", "pycocotools", "--user"])

In [ ]:
# Cell 2
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import json
import numpy as np
from PIL import Image
import os

COCO_JSON_PATH = "outputs/coco_dataset/annotations_coco.json"
IMAGE_DIR = "outputs/coco_dataset/images/"

class CocoMaskDataset(Dataset):
    def __init__(self, coco_json_path, image_dir):
        with open(coco_json_path) as f:
            self.coco = json.load(f)
        self.image_dir = image_dir
        self.images = {img['id']: img for img in self.coco['images']}
        self.annotations_by_image = {}
        for ann in self.coco['annotations']:
            self.annotations_by_image.setdefault(ann['image_id'], []).append(ann)
        self.image_ids = list(self.images.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]
        img = Image.open(self.image_dir + img_info['file_name']).convert("RGB")
        img_tensor = T.ToTensor()(img)

        anns = self.annotations_by_image.get(img_id, [])
        boxes, labels, masks = [], [], []

        for ann in anns:
            x, y, w, h = ann['bbox']
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])

            mask = Image.new('L', (img_info['width'], img_info['height']), 0)
            from PIL import ImageDraw
            draw = ImageDraw.Draw(mask)
            for seg in ann['segmentation']:
                points = list(zip(seg[0::2], seg[1::2]))
                draw.polygon(points, fill=1)
            masks.append(np.array(mask, dtype=np.uint8))

        if len(boxes) == 0:
            boxes = [[0, 0, 1, 1]]
            labels = [1]
            masks = [np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)]

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'masks': torch.tensor(np.array(masks), dtype=torch.uint8),
            'image_id': torch.tensor([img_id])
        }
        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))

dataset = CocoMaskDataset(COCO_JSON_PATH, IMAGE_DIR)
print(f"✅ Dataset loaded: {len(dataset)} images")

img, target = dataset[0]
print(f"Image shape: {img.shape}")
print(f"Boxes: {target['boxes'].shape}")
print(f"Labels: {target['labels']}")
print(f"Masks shape: {target['masks'].shape}")

In [ ]:
# Cell 3 - Train/Val Split
from torch.utils.data import random_split

n_total = len(dataset)
n_val = max(3, int(n_total * 0.2))
n_train = n_total - n_val

train_dataset, val_dataset = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Cell 4 - Build Model
NUM_CLASSES = 16  # 15 LULC classes + 1 background

model = maskrcnn_resnet50_fpn(weights="DEFAULT")

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, NUM_CLASSES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
if device.type == "cpu":
    print("⚠️ CPU detected - this will be slow.")

model.to(device)
print(f"✅ Mask R-CNN ready, {NUM_CLASSES} classes")

In [ ]:
import time

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)

NUM_EPOCHS = 8

model.train()
start = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    avg_loss = epoch_loss / len(train_loader)
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | {elapsed:.0f}s elapsed")

print(f"\n✅ Training complete in {time.time()-start:.0f}s")
torch.save(model.state_dict(), "outputs/maskrcnn_shillong_lulc.pth")
print("✅ Model saved: outputs/maskrcnn_shillong_lulc.pth")

Labelling Of Unlabeled Patches With Labeled Patch

In [ ]:
# Quick check of the structure
print(type(dataset.images))
sample_key = list(dataset.images.keys())[0]
print(f"Sample key: {sample_key}")
print(f"Sample value: {dataset.images[sample_key]}")

In [ ]:
import os

ALL_PATCHES_DIR = "outputs/dataset/train/"

# Get the 25 already-labeled filenames using the CORRECT key
labeled_filenames = set(img_info['file_name'] for img_info in dataset.images.values())
print(f"Already labeled (exclude these): {len(labeled_filenames)}")

# Get unlabeled patches
all_patches = os.listdir(ALL_PATCHES_DIR)
unlabeled_patches = [f for f in all_patches if f not in labeled_filenames]
print(f"Total patches in pool: {len(all_patches)}")
print(f"Unlabeled patches available: {len(unlabeled_patches)}")

In [ ]:
Labelling

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
import json

CONFIDENCE_THRESHOLD = 0.7  # only trust high-confidence predictions
model.eval()

pseudo_labels = {}
processed_count = 0
accepted_count = 0

# Process a manageable batch first (not all 8000 at once)
SAMPLE_SIZE = 500  # start with 500, scale up later if working well
import random
random.seed(42)
sample_to_process = random.sample(unlabeled_patches, min(SAMPLE_SIZE, len(unlabeled_patches)))

print(f"Processing {len(sample_to_process)} unlabeled patches...")

with torch.no_grad():
    for fname in sample_to_process:
        img_path = ALL_PATCHES_DIR + fname
        img = Image.open(img_path).convert("RGB")
        img_tensor = T.ToTensor()(img).to(device)

        prediction = model([img_tensor])[0]

        scores = prediction['scores'].cpu().numpy()
        boxes = prediction['boxes'].cpu().numpy()
        labels = prediction['labels'].cpu().numpy()
        masks = prediction['masks'].cpu().numpy()

        # Keep only high-confidence detections
        high_conf_idx = scores >= CONFIDENCE_THRESHOLD

        if high_conf_idx.sum() > 0:
            pseudo_labels[fname] = {
                'boxes': boxes[high_conf_idx].tolist(),
                'labels': labels[high_conf_idx].tolist(),
                'scores': scores[high_conf_idx].tolist(),
                'masks': masks[high_conf_idx].tolist()  # Note: large, consider saving separately
            }
            accepted_count += 1

        processed_count += 1
        if processed_count % 50 == 0:
            print(f"  Processed {processed_count}/{len(sample_to_process)} | Accepted: {accepted_count}")

print(f"\n✅ Pseudo-labeling complete")
print(f"   Processed: {processed_count}")
print(f"   Accepted (high confidence): {accepted_count}")
print(f"   Rejected (low confidence): {processed_count - accepted_count}")

In [ ]:
OTHER FIX METHOD

In [ ]:
import shutil
import random
import os

# Pick a batch to send to Colab
random.seed(42)
batch_to_process = random.sample(unlabeled_patches, 1000)

os.makedirs("outputs/colab_batch/", exist_ok=True)
for fname in batch_to_process:
    shutil.copy(f"outputs/dataset/train/{fname}", f"outputs/colab_batch/{fname}")

# Create the zip with a clear, descriptive name
shutil.make_archive("outputs/unlabeled_batch_1000", 'zip', "outputs/colab_batch/")
print("✅ Zip ready: outputs/unlabeled_batch_1000.zip")
print(f"Contains {len(batch_to_process)} patches")

In [ ]:
import os
import json

ALL_PATCHES_DIR = "outputs/dataset/train/"

# Get the 25 already-labeled filenames from the COCO JSON
with open("outputs/coco_dataset/annotations_coco.json") as f:
    coco_data = json.load(f)

labeled_filenames = set(img['file_name'] for img in coco_data['images'])
print(f"Already labeled (exclude these): {len(labeled_filenames)}")

# Get unlabeled patches
all_patches = os.listdir(ALL_PATCHES_DIR)
unlabeled_patches = [f for f in all_patches if f not in labeled_filenames]
print(f"Total patches in pool: {len(all_patches)}")
print(f"Unlabeled patches available: {len(unlabeled_patches)}")

In [ ]:
import shutil
import random

random.seed(42)
batch_to_process = random.sample(unlabeled_patches, 1000)

os.makedirs("outputs/colab_batch/", exist_ok=True)
for fname in batch_to_process:
    shutil.copy(f"outputs/dataset/train/{fname}", f"outputs/colab_batch/{fname}")

shutil.make_archive("outputs/unlabeled_batch_1000", 'zip', "outputs/colab_batch/")
print("✅ Zip ready: outputs/unlabeled_batch_1000.zip")
print(f"Contains {len(batch_to_process)} patches")

In [ ]:
import json
import os

print("File exists:", os.path.exists("outputs/pseudo_labels.json"))

with open("outputs/pseudo_labels.json") as f:
    pseudo_labels = json.load(f)

print(f"Pseudo-labeled patches: {len(pseudo_labels)}")

In [ ]:
import json
from PIL import Image

PATCHES_DIR = "outputs/dataset/train/"

# Load existing COCO annotations (your 25 real labeled patches)
with open("outputs/coco_dataset/annotations_coco.json") as f:
    coco_data = json.load(f)

print(f"Starting with {len(coco_data['images'])} real labeled images")
print(f"Starting with {len(coco_data['annotations'])} real annotations")

# Find current max IDs to continue numbering from there
next_image_id = max(img['id'] for img in coco_data['images']) + 1
next_ann_id = max(ann['id'] for ann in coco_data['annotations']) + 1

# Add pseudo-labeled images and annotations
added_images = 0
added_annotations = 0

for fname, data in pseudo_labels.items():
    img_path = PATCHES_DIR + fname
    if not os.path.exists(img_path):
        continue

    with Image.open(img_path) as im:
        width, height = im.size

    coco_data['images'].append({
        "id": next_image_id,
        "file_name": fname,
        "width": width,
        "height": height
    })

    for box, label, score in zip(data['boxes'], data['labels'], data['scores']):
        x1, y1, x2, y2 = box
        bbox = [x1, y1, x2 - x1, y2 - y1]
        area = (x2 - x1) * (y2 - y1)

        # Approximate segmentation as bbox rectangle 
        # (since we didn't transfer masks, only boxes)
        segmentation = [[x1, y1, x2, y1, x2, y2, x1, y2]]

        coco_data['annotations'].append({
            "id": next_ann_id,
            "image_id": next_image_id,
            "category_id": label,
            "bbox": bbox,
            "area": area,
            "segmentation": segmentation,
            "iscrowd": 0,
            "is_pseudo_label": True,  # flag for transparency
            "confidence_score": score
        })
        next_ann_id += 1
        added_annotations += 1

    next_image_id += 1
    added_images += 1

print(f"\n✅ Added {added_images} pseudo-labeled images")
print(f"✅ Added {added_annotations} pseudo-labeled annotations")
print(f"\nFINAL COMBINED DATASET:")
print(f"   Total images:      {len(coco_data['images'])}")
print(f"   Total annotations: {len(coco_data['annotations'])}")

# Save combined dataset
with open("outputs/coco_dataset/annotations_coco_SEMISUPERVISED.json", "w") as f:
    json.dump(coco_data, f, indent=2)

print(f"\n✅ Saved: outputs/coco_dataset/annotations_coco_SEMISUPERVISED.json")

In [ ]:
import shutil
import os
import json

PATCHES_DIR = "outputs/dataset/train/"

with open("outputs/pseudo_labels.json") as f:
    pseudo_labels = json.load(f)

copied = 0
for fname in pseudo_labels.keys():
    src = PATCHES_DIR + fname
    dst = "outputs/coco_dataset/images/" + fname
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        copied += 1

print(f"✅ Copied {copied} pseudo-labeled images to dataset folder")

In [ ]:
import shutil

shutil.make_archive("outputs/semisupervised_images", 'zip', "outputs/coco_dataset/images/")
print("✅ Created: outputs/semisupervised_images.zip")

img_count = len(os.listdir("outputs/coco_dataset/images/"))
print(f"Contains {img_count} images")

In [ ]:
import os

print("Total images in coco_dataset/images/:", len(os.listdir("outputs/coco_dataset/images/")))
print("Zip file exists:", os.path.exists("outputs/semisupervised_images.zip"))
print("Zip file size (MB):", os.path.getsize("outputs/semisupervised_images.zip") / (1024*1024))

In [ ]:
import numpy as np
from PIL import Image
import os
import random

PATCH_DIR = "outputs/dataset/train/"
all_patches = os.listdir(PATCH_DIR)

print("Scanning for cloud-like patches...")

cloud_candidates = []
random.seed(42)
sample_to_check = random.sample(all_patches, 500)

for fname in sample_to_check:
    img = np.array(Image.open(PATCH_DIR + fname).convert("RGB"))
    brightness = img.mean()
    variance = img.std()
    
    if brightness > 150 and variance < 40:
        cloud_candidates.append((fname, brightness, variance))

print(f"Found {len(cloud_candidates)} candidates")

CLOUDS VERFICATION

MODEL ENHANCEMENT

In [ ]:
import matplotlib.pyplot as plt

n_show = min(12, len(cloud_candidates))
fig, axes = plt.subplots(3, 4, figsize=(16,12))

for ax, (fname, b, v) in zip(axes.flatten(), cloud_candidates[:n_show]):
    img = Image.open(PATCH_DIR + fname)
    ax.imshow(img)
    ax.set_title(f"{fname[:20]}\nb={b:.0f} v={v:.0f}", fontsize=8)
    ax.axis('off')

plt.suptitle('Candidate Cloud/No-Data Patches', fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/cloud_candidates_preview.png", dpi=100)
plt.show()

In [ ]:
known_cloud_patches = [
    "qwerty (1)_enhanced_patch_1792_1024.png",
    "qwerty (32)_enhanced_patch_1792_0256.png", 
    "qwerty (38)_enhanced_patch_1792_1024.png",
]
# (the RA306OCT and RA314OCT ones are in qgis_to_label, not dataset/train, 
#  so handle separately if needed)

QGIS_LABEL_DIR = "outputs/qgis_to_label/"
for fname in known_cloud_patches:
    path = QGIS_LABEL_DIR + fname
    if os.path.exists(path):
        img = Image.open(path)
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.title(fname[:30])
        plt.axis('off')
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(3, 3, figsize=(14,14))

for ax, (fname, b, v) in zip(axes.flatten(), cloud_candidates):
    img = Image.open(PATCH_DIR + fname)
    ax.imshow(img)
    ax.set_title(f"{fname[:25]}\nbright={b:.0f} var={v:.0f}", fontsize=8)
    ax.axis('off')

plt.suptitle('9 Cloud Candidates - Visual Verification', fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/cloud_candidates_9.png", dpi=110)
plt.show()

In [ ]:
import json

# Simple whole-image labels for cloud patches (class_id = 16)
cloud_annotations = []
CLOUD_CLASS_ID = 16

for fname, b, v in cloud_candidates:
    img = Image.open(PATCH_DIR + fname)
    width, height = img.size
    
    cloud_annotations.append({
        "filename": fname,
        "class_id": CLOUD_CLASS_ID,
        "bbox": [0, 0, width, height],  # whole image = cloud
        "width": width,
        "height": height
    })

print(f"✅ Prepared {len(cloud_annotations)} cloud annotations")

with open("outputs/cloud_annotations.json", "w") as f:
    json.dump(cloud_annotations, f, indent=2)

In [ ]:
CLASS_NAMES_UPDATED = {
    1: "Cropland", 2: "Fallow", 3: "Plantation", 4: "Sandy_area",
    5: "Scrub_land", 6: "Mining", 7: "Rural", 8: "Urban",
    9: "Deciduous", 10: "Evergreen", 11: "Forest_plantation",
    12: "Grass_grazing", 13: "Inland_wetland", 14: "Riverstream_canals",
    15: "Water_bodies", 16: "Cloud_NoData"
}
print("✅ Updated class names with Cloud_NoData (id 16)")

Merging This Colouds In The Existing Dataset

In [ ]:
import json
import shutil
import os

# Load existing semi-supervised dataset
with open("outputs/coco_dataset/annotations_coco_SEMISUPERVISED.json") as f:
    coco_data = json.load(f)

# Load cloud annotations we just prepared
with open("outputs/cloud_annotations.json") as f:
    cloud_annotations = json.load(f)

print(f"Before: {len(coco_data['images'])} images, {len(coco_data['annotations'])} annotations")

# Add new category
coco_data['categories'].append({
    "id": 16, "name": "Cloud_NoData", "supercategory": "land_cover"
})

next_image_id = max(img['id'] for img in coco_data['images']) + 1
next_ann_id = max(ann['id'] for ann in coco_data['annotations']) + 1

PATCH_DIR = "outputs/dataset/train/"

for cloud_ann in cloud_annotations:
    fname = cloud_ann['filename']
    
    coco_data['images'].append({
        "id": next_image_id,
        "file_name": fname,
        "width": cloud_ann['width'],
        "height": cloud_ann['height']
    })
    
    x, y, w, h = cloud_ann['bbox']
    segmentation = [[x, y, x+w, y, x+w, y+h, x, y+h]]
    
    coco_data['annotations'].append({
        "id": next_ann_id,
        "image_id": next_image_id,
        "category_id": 16,
        "bbox": [x, y, w, h],
        "area": w * h,
        "segmentation": segmentation,
        "iscrowd": 0
    })
    
    next_image_id += 1
    next_ann_id += 1

print(f"After: {len(coco_data['images'])} images, {len(coco_data['annotations'])} annotations")

with open("outputs/coco_dataset/annotations_coco_WITH_CLOUD.json", "w") as f:
    json.dump(coco_data, f, indent=2)

print("✅ Saved: outputs/coco_dataset/annotations_coco_WITH_CLOUD.json")

In [ ]:
copied = 0
for cloud_ann in cloud_annotations:
    fname = cloud_ann['filename']
    src = PATCH_DIR + fname
    dst = "outputs/coco_dataset/images/" + fname
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        copied += 1

print(f"✅ Copied {copied} cloud images to dataset folder")

total_images = len(os.listdir("outputs/coco_dataset/images/"))
print(f"Total images now in dataset folder: {total_images}")

In [ ]:
shutil.make_archive("outputs/dataset_with_cloud", 'zip', "outputs/coco_dataset/images/")
print("✅ Created: outputs/dataset_with_cloud.zip")
print(f"Size: {os.path.getsize('outputs/dataset_with_cloud.zip')/(1024*1024):.1f} MB")

In [ ]:
import os
import json

# Verify the merged JSON
with open("outputs/coco_dataset/annotations_coco_WITH_CLOUD.json") as f:
    coco_data = json.load(f)

print(f"Total images: {len(coco_data['images'])}")
print(f"Total annotations: {len(coco_data['annotations'])}")
print(f"Total categories: {len(coco_data['categories'])}")

# Check class 16 specifically made it in
cloud_anns = [a for a in coco_data['annotations'] if a['category_id'] == 16]
print(f"\nCloud_NoData (class 16) annotations: {len(cloud_anns)}")

# Verify images folder count matches
img_count = len(os.listdir("outputs/coco_dataset/images/"))
print(f"\nImages in folder: {img_count}")

TESTING PHASESESSS

In [ ]:
import os
import json

# All patches ever used in training (real + pseudo-labeled + cloud)
with open("outputs/coco_dataset/annotations_coco_WITH_CLOUD.json") as f:
    coco_data = json.load(f)

used_filenames = set(img['file_name'] for img in coco_data['images'])
print(f"Total patches used across all training stages: {len(used_filenames)}")

# Full pool
ALL_PATCHES_DIR = "outputs/dataset/train/"
all_patches = set(os.listdir(ALL_PATCHES_DIR))
print(f"Total patches in full pool: {len(all_patches)}")

# Genuinely unused = never touched at any stage
never_used = list(all_patches - used_filenames)
print(f"Never-used patches available for TRUE test set: {len(never_used)}")

SELECT TESTSSSS SAMPLESSS

In [ ]:
import random
import shutil

random.seed(100)  # different seed from all previous sampling
TEST_SAMPLE_SIZE = 15

test_patches = random.sample(never_used, TEST_SAMPLE_SIZE)

os.makedirs("outputs/final_test_set/", exist_ok=True)
for fname in test_patches:
    shutil.copy(ALL_PATCHES_DIR + fname, f"outputs/final_test_set/{fname}")

print(f"✅ {len(test_patches)} fresh test patches selected")
for f in test_patches:
    print(f"  {f}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(3, 5, figsize=(20,12))
for ax, fname in zip(axes.flatten(), test_patches):
    img = Image.open(f"outputs/final_test_set/{fname}")
    ax.imshow(img)
    ax.set_title(fname[:25], fontsize=7)
    ax.axis('off')

plt.suptitle('Step 6: Fresh Test Set (Never Seen by Model)', fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/final_test_set_preview.png", dpi=100)
plt.show()

In [ ]:
shutil.make_archive("outputs/final_test_set", 'zip', "outputs/final_test_set/")
print("✅ Created: outputs/final_test_set.zip")
print(f"Size: {os.path.getsize('outputs/final_test_set.zip')/1024:.1f} KB")